# 1. Atlas processing: QuPath exports to a region x mouse density matrix

Turns the per-image QuPath annotation exports into the two tables every
downstream notebook reads.

**Input**
- `data/raw_annotations/*.csv` -- one annotation-measurement CSV per mouse,
  written by `qupath/03_export_measurements.groovy`. Each row is one ABBA
  atlas annotation on one section, carrying its area, cFos+ count, NeuN
  channel mean, and `Exclude` flag.
- The Allen CCFv3 structure graph, fetched live from the Allen Brain Atlas
  RMA API (needs a network connection on first run).

**Output**, written to `data/compiled/`
- `density_by_region_batch.csv` -- every Allen region, all ontology levels,
  one column per mouse. Densities are cells/mm2.
- `cleaned_output/density_by_region_leaves_wide.csv` -- regions x mice, after
  the three exclusion passes below.
- `cleaned_output/density_by_region_leaves_long.csv` -- the same table in long
  format, one row per (region, mouse), which is what the per-region LMMs want.

**What the cleaning does.** Three passes, in a fixed order that matters:
drop the non-neural top-level divisions, drop numbered cortical layer
subdivisions, then drop any region whose children are all individually present.
Ancestry has to be resolved last, because which rows count as leaves depends on
what the first two passes removed.

Sex, genotype, light condition and batch are **not** attached here -- that
metadata lives in `data/mouse_info_template.csv` and is merged in by each
analysis notebook, keyed on `mouse_id`.


In [ ]:
# LOAD PACKAGES
import json
import urllib.request
from pathlib import Path

import pandas as pd

# ---------------------------------------------------------------------------
# Paths -- repo-relative, so this notebook runs wherever the repository is
# checked out. PROJECT_ROOT is found by walking up from the working directory
# until a folder containing `data/` appears. If you keep the data outside the
# repository, set PROJECT_ROOT by hand and everything below follows.
# ---------------------------------------------------------------------------
def find_project_root(markers=("data", "notebooks")):
    """Nearest ancestor directory containing all of `markers`. Requiring both
    `data/` and `notebooks/` means a stray `data/` folder somewhere on the path
    cannot be mistaken for the repository root."""
    here = Path.cwd().resolve()
    for candidate in (here, *here.parents):
        if all((candidate / m).is_dir() for m in markers):
            return candidate
    raise RuntimeError(
        f"No ancestor of {here} contains {list(markers)}. Set PROJECT_ROOT by hand."
    )

PROJECT_ROOT = find_project_root()
DATA_DIR     = PROJECT_ROOT / "data"
RESULTS_DIR  = PROJECT_ROOT / "results"

RAW_ANNOTATION_DIR = DATA_DIR / "raw_annotations"   # QuPath per-mouse exports
COMPILED_DIR       = DATA_DIR / "compiled"          # where this notebook writes
COMPILED_DIR.mkdir(parents=True, exist_ok=True)


In [ ]:
def process_density_data_batch(input_folder, output_folder):
    import os
    import json
    import urllib.request
    import pandas as pd

    os.makedirs(output_folder, exist_ok=True)

    # ===== FIND INPUT FILES =====
    input_files = [f for f in os.listdir(input_folder) if f.endswith('.csv')]
    print(f"Found {len(input_files)} CSV files: {input_files}")

    # ===== GET ALLEN ATLAS HIERARCHY =====
    print("\nFetching Allen Atlas hierarchy...")
    url = 'http://api.brain-map.org/api/v2/structure_graph_download/1.json'
    with urllib.request.urlopen(url) as response:
        structure_graph = json.loads(response.read())

    def flatten_with_ancestors(node, ancestors=None, result=None):
        if result is None:
            result = []
        if ancestors is None:
            ancestors = []
        current_ancestors = ancestors + [node['name']]
        result.append({
            'name': node['name'],
            'acronym': node.get('acronym', ''),
            'structure_id': node['id'],
            'hierarchy_order': len(result),
            'depth': len(ancestors),
            'ancestors': current_ancestors
        })
        for child in node.get('children', []):
            flatten_with_ancestors(child, current_ancestors, result)
        return result

    hierarchy = pd.DataFrame(flatten_with_ancestors(structure_graph['msg'][0]))

    max_depth = hierarchy['depth'].max()
    level_names = [f'Level_{i}' for i in range(max_depth + 1)]
    ancestor_df = hierarchy['ancestors'].apply(
        lambda x: pd.Series(x + [None] * (max_depth + 1 - len(x)))
    )
    ancestor_df.columns = level_names
    hierarchy = pd.concat([hierarchy, ancestor_df], axis=1)

    # ===== PROCESS EACH FILE =====
    all_densities = {}

    for filename in sorted(input_files):
        file_path = os.path.join(input_folder, filename)
        sample_id = os.path.splitext(filename)[0]
        print(f"\nProcessing: {filename}")

        df = pd.read_csv(file_path)
        df.columns = df.columns.str.strip()

        # ===== EXCLUDE FLAGGED ANNOTATIONS =====
        exclude_col = [c for c in df.columns if c.lower() == 'exclude']
        if exclude_col:
            n_before = len(df)
            df = df[df[exclude_col[0]] != 1]
            n_excluded = n_before - len(df)
            if n_excluded > 0:
                print(f"  Excluded {n_excluded} flagged annotation(s)")
        else:
            print(f"  Warning: no 'Exclude' column found in {filename}")

        name_col = 'Name'
        area_col = [c for c in df.columns if 'area' in c.lower() or 'µm' in c.lower()][0]
        count_col = [c for c in df.columns if 'positive' in c.lower() or 'num' in c.lower()][0]

        grouped = df.groupby(name_col).agg(
            total_area_um2=(area_col, 'sum'),
            total_detections=(count_col, 'sum')
        ).reset_index()

        grouped['density_cells_per_mm2'] = (grouped['total_detections'] / grouped['total_area_um2']) * 1e6

        all_densities[sample_id] = grouped.set_index(name_col)['density_cells_per_mm2']

        print(f"  {len(grouped)} regions retained")

    # ===== COMBINE INTO WIDE FORMAT =====
    density_wide = pd.DataFrame(all_densities)
    density_wide.index.name = 'Acronym'
    density_wide = density_wide.reset_index()

    # ===== MERGE WITH HIERARCHY =====
    merge_cols = ['acronym', 'name', 'hierarchy_order'] + level_names
    density_wide = density_wide.merge(
        hierarchy[merge_cols],
        left_on='Acronym', right_on='acronym', how='left'
    ).drop(columns='acronym')

    density_wide = density_wide.rename(columns={'name': 'Brain Region'})

    unmatched = density_wide[density_wide['hierarchy_order'].isna()]['Acronym'].tolist()
    if unmatched:
        print(f"\nWarning: {len(unmatched)} regions didn't match Allen hierarchy:")
        for r in unmatched:
            print(f"  - {r}")

    density_wide = density_wide.sort_values('hierarchy_order', na_position='last')

    # ===== FORMAT OUTPUT =====
    used_levels = [c for c in level_names if density_wide[c].notna().any()]
    sample_cols = sorted(all_densities.keys())

    output_cols = ['Acronym', 'Brain Region'] + used_levels + sample_cols
    output = density_wide[output_cols].copy()
    output[sample_cols] = output[sample_cols].round(2)

    # ===== SAVE =====
    output_path = os.path.join(output_folder, 'density_by_region_batch.csv')
    output.to_csv(output_path, index=False)
    print(f"\nSaved batch output to: {output_path}")
    print(f"Shape: {len(output)} regions × {len(sample_cols)} samples")

    return output

## 2. Region selection: drop non-neural divisions, cortical layers, and redundant ancestors

In [ ]:
df = process_density_data_batch(
    input_folder=RAW_ANNOTATION_DIR,
    output_folder=COMPILED_DIR,
)

In [ ]:
import re

INPUT_CSV  = COMPILED_DIR / "density_by_region_batch.csv"
OUTPUT_DIR = COMPILED_DIR / "cleaned_output"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

LEVEL_COLS = [f"Level_{i}" for i in range(10)]  # ancestry path, root -> leaf
REGION_NAME_COL = "Brain Region"
REGION_ACRONYM_COL = "Acronym"

MEASUREMENT_COL_FLAG = "annotation_measurements"

# ### CHOICE ###
# "grey" (Basic cell groups and regions) is the only Level_1 division retained.
# "fiber tracts" and "ventricular systems" are excluded wholesale -- neither
# contains cell bodies, so cFos+ counts/density there aren't biologically
# interpretable the same way and would just add noise/multiple-comparison
# burden to the region-wise LMMs.
EXCLUDE_TOP_LEVEL_DIVISIONS = {"fiber tracts", "ventricular systems"}

In [ ]:
def flag_excluded_divisions(df: pd.DataFrame, level_col: str = "Level_1") -> pd.Series:
    """
    Flag rows belonging to non-neural top-level divisions (fiber tracts,
    ventricular systems) for exclusion. These sit at Level_1, as siblings of
    "grey" (Basic cell groups and regions), so a single membership check on
    Level_1 correctly catches the division itself and every one of its
    descendants in one pass.
    """
    return df[level_col].isin(EXCLUDE_TOP_LEVEL_DIVISIONS)


def flag_cortical_layers(df: pd.DataFrame, name_col: str = REGION_NAME_COL) -> pd.Series:
    """
    Identify rows that are numbered isocortex/allocortex layer subdivisions
    (e.g. 'Primary motor area, Layer 2/3', 'Frontal pole, layer 6b'), as opposed
    to the parent area (e.g. 'Primary motor area') or non-cortical laminae that
    happen to contain the word "layer" (e.g. 'Dentate gyrus, molecular layer',
    'Superior colliculus, optic layer' -- these are distinct cytoarchitectonic
    structures, not redundant subdivisions of a cortical area, so they are kept).

    ### CHOICE ###
    A region is flagged as a "layer" row only if its name ends in a recognized
    numbered-layer token (1, 2/3, 3, 4, 5, 6a, 6b), optionally preceded by the
    word "layer". This is deliberately narrower than matching on the substring
    "layer" alone, for the reason above.

    Two source-naming conventions are both handled:
      1. "<Area>, layer <N>"      e.g. "Secondary motor area, layer 2/3"
      2. "<Area>/Layer <N>"       e.g. "Ectorhinal area/Layer 1"
    A third quirk in the Allen ontology export omits the word "layer" for two
    specific regions ("Anterior cingulate area, ventral part, 6a"/"6b") -- this
    is caught separately.
    """
    numbered_layer = re.compile(r"[,/]\s*layer\s*(1|2/3|3|4|5|6a|6b)\s*$", re.IGNORECASE)
    bare_numbered_layer = re.compile(r",\s*(1|2/3|3|4|5|6a|6b)\s*$")  # "..., 6a" with no "layer" word

    names = df[name_col].fillna("")
    is_layer = names.str.contains(numbered_layer) | names.str.contains(bare_numbered_layer)
    return is_layer


def get_ancestry_path(row: pd.Series, level_cols: list = LEVEL_COLS) -> tuple:
    """Return this row's full ancestry path (root -> self) as a tuple of names,
    dropping empty/NaN levels. The row's own name is the last element."""
    return tuple(row[c] for c in level_cols if pd.notna(row[c]))


def flag_ancestor_regions(df: pd.DataFrame, level_cols: list = LEVEL_COLS) -> pd.Series:
    """
    Identify rows that are strict ancestors of another row already present in
    `df`. If every child of a region is present individually (e.g. all thalamic
    nuclei), the parent ("Thalamus") is redundant for region-wise stats and is
    flagged for removal here.

    ### CHOICE ###
    Run this LAST, after both the fiber-tract/ventricle exclusion and the
    cortical-layer exclusion -- ancestry status depends on which rows are still
    present, so it has to see the fully-trimmed candidate set to correctly
    resolve e.g. cortical areas (FRP, MOp) into leaves once their layers are gone.
    """
    paths = df.apply(get_ancestry_path, axis=1, level_cols=level_cols)
    path_list = paths.tolist()

    is_ancestor = []
    for path in path_list:
        flagged = any(
            other != path and len(other) > len(path) and other[: len(path)] == path
            for other in path_list
        )
        is_ancestor.append(flagged)
    return pd.Series(is_ancestor, index=df.index)


def get_measurement_cols(df: pd.DataFrame, flag: str = MEASUREMENT_COL_FLAG) -> list:
    return [c for c in df.columns if flag in c]

## 3. Reshape to long format and write both tables

In [ ]:
raw = pd.read_csv(INPUT_CSV)
meas_cols = get_measurement_cols(raw)
print(f"Loaded {len(raw)} regions x {len(meas_cols)} mice")

# Step 1: drop fiber tracts / ventricular system
is_excluded_div = flag_excluded_divisions(raw)
neural_only = raw.loc[~is_excluded_div].copy()
print(f"\nDropped {is_excluded_div.sum()} fiber-tract/ventricle rows -> {len(neural_only)} remaining")

# Step 2: drop cortical layer subdivisions
is_layer = flag_cortical_layers(neural_only)
no_layers = neural_only.loc[~is_layer].copy()
print(f"Dropped {is_layer.sum()} cortical-layer rows -> {len(no_layers)} remaining")

# Step 3: drop ancestor regions whose children are all individually present
is_ancestor = flag_ancestor_regions(no_layers)
leaves = no_layers.loc[~is_ancestor].copy()
print(f"Dropped {is_ancestor.sum()} ancestor rows -> {len(leaves)} leaf regions remaining")

# Sanity spot-checks -- adjust/remove as you like
print("\nExample dropped fiber-tract/ventricle rows:")
print(raw.loc[is_excluded_div, [REGION_ACRONYM_COL, REGION_NAME_COL]].head(6).to_string(index=False))

print("\nExample dropped layer rows:")
print(neural_only.loc[is_layer, [REGION_ACRONYM_COL, REGION_NAME_COL]].head(6).to_string(index=False))

print("\nExample dropped ancestor rows:")
print(no_layers.loc[is_ancestor, [REGION_ACRONYM_COL, REGION_NAME_COL]].head(6).to_string(index=False))

In [ ]:
# ### CHOICE ###
# LMMs want long format: one row per (region, mouse). Mouse IDs are recovered
# by stripping the "_annotation_measurements" suffix from each column name.
# This does NOT attach sex/genotype/light-condition/batch metadata -- merge
# that in separately from your animal metadata table, keyed on mouse_id.
id_cols = [REGION_ACRONYM_COL, REGION_NAME_COL] + LEVEL_COLS
long_df = leaves.melt(
    id_vars=id_cols,
    value_vars=meas_cols,
    var_name="mouse_id",
    value_name="density",
)
long_df["mouse_id"] = long_df["mouse_id"].str.replace(f"_{MEASUREMENT_COL_FLAG}", "", regex=False)

print(f"Long-format table: {long_df.shape[0]} rows ({len(leaves)} regions x {len(meas_cols)} mice)")
print(f"Missing density values: {long_df['density'].isna().sum()} "
      f"({long_df['density'].isna().mean():.1%})")

leaves.to_csv(OUTPUT_DIR / "density_by_region_leaves_wide.csv", index=False)
long_df.to_csv(OUTPUT_DIR / "density_by_region_leaves_long.csv", index=False)

print(f"\nSaved:")
print(f"  {OUTPUT_DIR / 'density_by_region_leaves_wide.csv'}")
print(f"  {OUTPUT_DIR / 'density_by_region_leaves_long.csv'}")